# Notebook 04 — Physics-Aware Feature Ablation

Compare baseline (5 features) vs. +physics (8 features: 5 raw + Re, We, ρᵣ).

In [ ]:
# Requires: pip install CoolPropfrom CoolProp.CoolProp import PropsSIimport pandas as pdimport numpy as npdf = pd.read_parquet('../data/chf_clean.parquet')P_pa = df['Pressure'].values * 1000G = df['Mass Flux'].valuesD = df['Tube Diameter'].valuesrho_l = np.array([PropsSI('D', 'P', p, 'Q', 0, 'Water') for p in P_pa])rho_v = np.array([PropsSI('D', 'P', p, 'Q', 1, 'Water') for p in P_pa])mu_l  = np.array([PropsSI('V', 'P', p, 'Q', 0, 'Water') for p in P_pa])sigma = np.array([PropsSI('I', 'P', p, 'Q', 0, 'Water') for p in P_pa])df['Re']    = (G * D) / mu_ldf['We']    = (G**2 * D) / (rho_l * sigma)df['rho_r'] = rho_l / rho_vdf.to_parquet('../data/chf_with_physics.parquet', index=False)print('Physics features computed.')

In [ ]:
# Train both configurations and compareimport sys; sys.path.insert(0, '..')from src.splits import build_splitsfrom src.models import train_xgb_with_valfrom sklearn.metrics import r2_scoreFEATURES_BASE    = ['Tube Diameter', 'Heated Length', 'Pressure', 'Mass Flux', 'Outlet Quality']FEATURES_PHYSICS = FEATURES_BASE + ['Re', 'We', 'rho_r']y = df['CHF'].values.astype(np.float32)for cfg_name, features in [('Baseline', FEATURES_BASE), ('+Physics', FEATURES_PHYSICS)]:    X = df[features].values.astype(np.float32)    splits = build_splits(X, y, df, seed=42)    print(f'\n{cfg_name} ({len(features)} features):')    for split_name, (X_tr, y_tr, X_va, y_va, X_te, y_te) in splits.items():        m = train_xgb_with_val(X_tr, y_tr, X_va, y_va, seed=42)        r2 = r2_score(y_te, m.predict(X_te))        print(f'  {split_name}: R² = {r2:.4f}')